## Gym Synthetic Dataset Generator

### Overview

This notebook generates a synthetic dataset that simulates the operations of a gym membership business.

The goal is to create realistic behavioral and transactional data that can be used for analytics and machine learning tasks, particularly **customer churn prediction**.

The dataset models several aspects of gym operations, including:

- Member demographics
- Membership plans
- Gym attendance behavior
- Payment transactions
- Membership cancellations (churn)

Synthetic data allows us to simulate realistic business scenarios while maintaining full control over the data generation process.

---

### Generated Datasets

This notebook produces three datasets:

#### Members
Contains demographic and membership information.

| Column | Description |
|------|-------------|
member_id | Unique identifier for each member |
age | Member age |
gender | Member gender |
city | City where the member is located |
signup_channel | Marketing acquisition channel |
plan_type | Membership plan type |
engagement_level | Simulated engagement profile |
join_date | Membership start date |
cancel_date | Membership cancellation date |

---

#### Checkins
Simulates gym attendance behavior.

| Column | Description |
|------|-------------|
member_id | Member identifier |
date | Date of gym visit |
hour | Hour of the visit |

The simulation includes realistic patterns such as:

- Different training frequencies
- Random training breaks
- Declining engagement before churn

---

#### Payments
Represents membership payment transactions.

| Column | Description |
|------|-------------|
member_id | Member identifier |
date | Payment date |
amount | Payment amount |
plan_type | Membership plan |

Payments follow billing cycles depending on the membership plan:

- Monthly
- Quarterly
- Annual

---

### Purpose

The generated datasets are used in the **analysis notebook** to:

- Explore member behavior
- Perform feature engineering
- Build machine learning models
- Predict customer churn

---

### Output Files

The notebook exports the following files:
- members.csv
- checkins.csv
- to_csv

In [1]:
"""
Gym Synthetic Dataset Generator
--------------------------------

This notebook generates a synthetic dataset simulating
the operations of a gym membership business.

The generated data includes:

- Member demographic information
- Gym check-in behavior
- Payment transactions

The dataset is designed to support analytics and
machine learning tasks such as churn prediction.
"""

# ============================================
# ENVIRONMENT SETUP
# ============================================

import pandas as pd
import numpy as np

import random

from datetime import timedelta


pd.set_option("display.max_columns", None)

print("Environment successfully configured.")

Environment successfully configured.


In [2]:
"""
Simulation Parameters
---------------------

This section defines the main parameters used to
generate the synthetic gym dataset.

Adjusting these values allows us to simulate
different business sizes and behaviors.
"""

# ============================================
# DATASET PARAMETERS
# ============================================

# Number of simulated gym members
N_MEMBERS = 5000


# Simulation period
START_DATE = pd.Timestamp("2018-01-01")
END_DATE = pd.Timestamp("2025-01-01")


# Membership plans and prices
PLAN_PRICES = {
    "monthly": 100,
    "quarterly": 270,
    "annual": 960
}


# Cities where the gym operates
CITIES = [
    "Florianópolis",
    "Rio de Janeiro",
    "São Paulo",
    "Curitiba",
    "Recife",
    "Porto Alegre"
]


# Marketing acquisition channels
SIGNUP_CHANNELS = [
    "organic",
    "instagram_ads",
    "google_ads",
    "referral",
    "walk_in"
]


print("Simulation parameters initialized.")

Simulation parameters initialized.


In [3]:
"""
Helper Functions
----------------

These utility functions support the generation of
synthetic dates and random behaviors.
"""

# ============================================
# HELPER FUNCTIONS
# ============================================

def generate_join_date():
    
    """
    Generate a random membership start date
    within the simulation period.
    """
    
    days_range = (END_DATE - START_DATE).days
    
    random_days = random.randint(0, days_range)
    
    return START_DATE + pd.Timedelta(days=random_days)

In [4]:
"""
Members Dataset Generation
--------------------------

This section generates the base dataset containing
demographic and membership information for each member.
"""

# ============================================
# GENERATE MEMBERS DATASET
# ============================================

members_list = []


for i in range(N_MEMBERS):
    
    member_id = i + 1
    
    # ----------------------------------------
    # AGE DISTRIBUTION
    # ----------------------------------------
    
    age_group = random.choices(
        ["young","mid","older"],
        weights=[0.6,0.25,0.15]
    )[0]
    
    
    if age_group == "young":
        age = random.randint(18,33)
        
    elif age_group == "mid":
        age = random.randint(34,45)
        
    else:
        age = random.randint(46,65)
    
    
    # ----------------------------------------
    # GENDER
    # ----------------------------------------
    
    gender = random.choice(["male","female"])
    
    
    # ----------------------------------------
    # CITY
    # ----------------------------------------
    
    city = random.choice(CITIES)
    
    
    # ----------------------------------------
    # SIGNUP CHANNEL
    # ----------------------------------------
    
    signup_channel = random.choices(
        SIGNUP_CHANNELS,
        weights=[0.35,0.2,0.2,0.15,0.1]
    )[0]
    
    
    # ----------------------------------------
    # MEMBERSHIP PLAN
    # ----------------------------------------
    
    plan_type = random.choices(
        ["monthly","quarterly","annual"],
        weights=[0.6,0.25,0.15]
    )[0]
    
    
    # ----------------------------------------
    # ENGAGEMENT LEVEL
    # ----------------------------------------
    
    engagement_level = random.choices(
        ["low","medium","high"],
        weights=[0.3,0.45,0.25]
    )[0]
    
    
    # ----------------------------------------
    # JOIN DATE
    # ----------------------------------------
    
    join_date = generate_join_date()
    
    
    members_list.append({
        "member_id":member_id,
        "age":age,
        "gender":gender,
        "city":city,
        "signup_channel":signup_channel,
        "plan_type":plan_type,
        "engagement_level":engagement_level,
        "join_date":join_date
    })


members = pd.DataFrame(members_list)

print("Members dataset generated.")
print("Total members:",len(members))

Members dataset generated.
Total members: 5000


In [5]:
"""
Churn Simulation
----------------

This section simulates membership cancellations.

Churn probability depends on:

- engagement level
- membership plan type
"""

# ============================================
# GENERATE CHURN
# ============================================

cancel_dates = []


for _,row in members.iterrows():
    
    join_date = row["join_date"]
    
    engagement = row["engagement_level"]
    
    plan = row["plan_type"]
    
    
    # Base churn probability
    
    if engagement == "low":
        churn_prob = 0.55
        
    elif engagement == "medium":
        churn_prob = 0.30
        
    else:
        churn_prob = 0.15
    
    
    if plan == "monthly":
        churn_prob += 0.10
        
    if plan == "annual":
        churn_prob -= 0.05
    
    
    churn = np.random.rand() < churn_prob
    
    
    if churn:
        
        min_days = 60
        
        max_days = (END_DATE - join_date).days
        
        if max_days <= min_days:
            cancel_dates.append(pd.NaT)
            continue
        
        cancel_after = random.randint(min_days,max_days)
        
        cancel_dates.append(
            join_date + pd.Timedelta(days=cancel_after)
        )
        
    else:
        
        cancel_dates.append(pd.NaT)



members["cancel_date"] = cancel_dates

print("Churn simulation completed.")

Churn simulation completed.


In [6]:
"""
Check-in Simulation
-------------------

This section generates gym attendance records for each member.

The simulation includes realistic training behavior:

• Weekly training patterns
• Engagement-based frequency
• Declining activity before churn
• Random breaks (vacation / illness)
• Typical gym training hours

This dataset will be used to analyze gym usage
and to build behavioral churn prediction features.
"""

# ============================================
# GENERATE CHECK-IN EVENTS
# ============================================

checkins_list = []


for _, member in members.iterrows():
    
    member_id = member["member_id"]
    
    join_date = member["join_date"]
    
    cancel_date = member["cancel_date"]
    
    engagement = member["engagement_level"]
    
    
    # Define the last active date
    
    end_date = cancel_date if pd.notna(cancel_date) else END_DATE
    
    
    current_date = join_date
    
    
    while current_date < end_date:
        
        
        # ------------------------------------
        # BASE WEEKLY TRAINING FREQUENCY
        # ------------------------------------
        
        if engagement == "low":
            
            base_visits = random.choices(
                [0,1,2],
                weights=[0.35,0.45,0.20]
            )[0]
        
        
        elif engagement == "medium":
            
            base_visits = random.choices(
                [2,3,4],
                weights=[0.30,0.45,0.25]
            )[0]
        
        
        else:
            
            base_visits = random.choices(
                [3,4,5,6],
                weights=[0.20,0.35,0.30,0.15]
            )[0]
        
        
        visits = base_visits
        
        
        # ------------------------------------
        # ENGAGEMENT DECAY BEFORE CHURN
        # ------------------------------------
        
        if pd.notna(cancel_date):
            
            weeks_to_churn = (cancel_date - current_date).days / 7
            
            
            if weeks_to_churn < 12:
                visits = max(0, int(visits * 0.6))
            
            if weeks_to_churn < 6:
                visits = max(0, int(visits * 0.4))
            
            if weeks_to_churn < 3:
                visits = max(0, int(visits * 0.2))
        
        
        # ------------------------------------
        # RANDOM TRAINING BREAKS
        # ------------------------------------
        
        # Members may skip training due to
        # travel, illness or work schedules.
        
        if random.random() < 0.03:
            
            current_date += pd.Timedelta(days=14)
            
            continue
        
        
        # ------------------------------------
        # GENERATE WEEKLY VISITS
        # ------------------------------------
        
        for _ in range(visits):
            
            
            # Gym opens Monday → Saturday
            
            weekday = random.choices(
                [0,1,2,3,4,5],
                weights=[24,22,20,15,11,8]
            )[0]
            
            
            week_start = current_date - pd.Timedelta(
                days=current_date.weekday()
            )
            
            
            visit_date = week_start + pd.Timedelta(days=weekday)
            
            
            if visit_date < end_date:
                
                
                # Typical gym hours
                
                hour = random.choices(
                    [6,7,8,9,10,11,12,13,14,15,16,17,18,19,20,21],
                    weights=[
                        10,12,10,
                        5,4,4,4,4,4,4,4,
                        8,
                        12,14,12,
                        6
                    ]
                )[0]
                
                
                checkins_list.append({
                    "member_id":member_id,
                    "date":visit_date,
                    "hour":hour
                })
        
        
        # Move to next week
        
        current_date += pd.Timedelta(days=7)



checkins = pd.DataFrame(checkins_list)


print("Check-ins dataset generated.")
print("Total check-ins:", len(checkins))

Check-ins dataset generated.
Total check-ins: 1892634


In [7]:
"""
Payment Simulation
------------------

This section generates the membership payment history
for each gym member.

The payment schedule depends on the membership plan:

• Monthly     → payment every 30 days
• Quarterly   → payment every 90 days
• Annual      → payment every 365 days

Payments stop when a member cancels their membership.

This dataset will later support revenue analysis
and lifetime value calculations.
"""

# ============================================
# GENERATE PAYMENTS DATASET
# ============================================

payments_list = []


for _, member in members.iterrows():
    
    member_id = member["member_id"]
    
    join_date = member["join_date"]
    
    cancel_date = member["cancel_date"]
    
    plan = member["plan_type"]
    
    
    # Determine the last billing date
    
    end_date = cancel_date if pd.notna(cancel_date) else END_DATE
    
    
    # ----------------------------------------
    # PAYMENT FREQUENCY
    # ----------------------------------------
    
    if plan == "monthly":
        
        billing_cycle = 30
        
    elif plan == "quarterly":
        
        billing_cycle = 90
        
    else:
        
        billing_cycle = 365
    
    
    price = PLAN_PRICES[plan]
    
    
    current_payment_date = join_date
    
    
    while current_payment_date < end_date:
        
        
        payments_list.append({
            
            "member_id": member_id,
            
            "date": current_payment_date,
            
            "amount": price,
            
            "plan_type": plan
            
        })
        
        
        # Move to next billing cycle
        
        current_payment_date += pd.Timedelta(days=billing_cycle)



payments = pd.DataFrame(payments_list)


print("Payments dataset generated.")
print("Total payments:", len(payments))

Payments dataset generated.
Total payments: 119587


In [8]:
"""
Dataset Validation
------------------

This section performs basic validation of the generated datasets.

The goal is to ensure that the synthetic data generation
process produced consistent and realistic outputs.

Validation includes:

• dataset dimensions
• preview of records
• descriptive statistics
"""

# ============================================
# DATASET SHAPES
# ============================================

print("Dataset shapes:\n")

print("Members:", members.shape)
print("Checkins:", checkins.shape)
print("Payments:", payments.shape)

Dataset shapes:

Members: (5000, 9)
Checkins: (1892634, 3)
Payments: (119587, 4)


In [9]:
# ============================================
# DATA PREVIEW
# ============================================

print("\nMembers preview:")
display(members.head())

print("\nCheckins preview:")
display(checkins.head())

print("\nPayments preview:")
display(payments.head())


Members preview:


,member_id,age,gender,city,signup_channel,plan_type,engagement_level,join_date,cancel_date
0,1,58,male,Recife,walk_in,monthly,medium,2021-10-03,2022-11-25
1,2,19,female,Curitiba,referral,monthly,high,2019-07-11,NaT
2,3,23,female,São Paulo,referral,quarterly,high,2023-11-29,2024-04-25
3,4,30,male,Recife,organic,quarterly,medium,2022-10-23,2023-01-22
4,5,33,female,Florianópolis,organic,monthly,low,2023-12-09,NaT



Checkins preview:


,member_id,date,hour
0,1,2021-09-28,18
1,1,2021-09-29,20
2,1,2021-09-30,10
3,1,2021-09-27,19
4,1,2021-10-04,21



Payments preview:


,member_id,date,amount,plan_type
0,1,2021-10-03,100,monthly
1,1,2021-11-02,100,monthly
2,1,2021-12-02,100,monthly
3,1,2022-01-01,100,monthly
4,1,2022-01-31,100,monthly


In [10]:
# ============================================
# BASIC STATISTICS
# ============================================

print("\nMembers statistics:")
display(members.describe(include="all"))

print("\nPayments statistics:")
display(payments.describe())


Members statistics:


,member_id,age,gender,city,signup_channel,plan_type,engagement_level,join_date,cancel_date
count,5000.000000,5000.000000,5000,5000,5000,5000,5000,5000,1983
unique,NaN,NaN,2,6,5,3,3,NaN,NaN
top,NaN,NaN,male,Curitiba,organic,monthly,medium,NaN,NaN
freq,NaN,NaN,2541,880,1704,2993,2224,NaN,NaN
mean,2500.500000,33.533800,NaN,NaN,NaN,NaN,NaN,2021-07-02 04:54:54.720000,2023-04-21 01:47:28.411497728
min,1.000000,18.000000,NaN,NaN,NaN,NaN,NaN,2018-01-01 00:00:00,2018-07-08 00:00:00
25%,1250.750000,24.000000,NaN,NaN,NaN,NaN,NaN,2019-10-21 12:00:00,2022-06-02 00:00:00
50%,2500.500000,31.000000,NaN,NaN,NaN,NaN,NaN,2021-06-30 12:00:00,2023-10-08 00:00:00
75%,3750.250000,41.000000,NaN,NaN,NaN,NaN,NaN,2023-03-24 06:00:00,2024-07-16 00:00:00
max,5000.000000,65.000000,NaN,NaN,NaN,NaN,NaN,2024-12-31 00:00:00,2025-01-01 00:00:00



Payments statistics:


,member_id,date,amount
count,119587.000000,119587,119587.000000
mean,2504.542735,2022-06-13 20:55:40.898258176,140.852099
min,1.000000,2018-01-01 00:00:00,100.000000
25%,1267.000000,2021-03-25 00:00:00,100.000000
50%,2486.000000,2022-09-02 00:00:00,100.000000
75%,3768.000000,2023-11-10 00:00:00,100.000000
max,5000.000000,2024-12-31 00:00:00,960.000000
std,1451.108672,NaN,135.631303


In [11]:
"""
Save Generated Datasets
-----------------------

This section exports the generated datasets
to CSV files so they can be used in the
analysis notebook.

The files are saved in the /data directory.
"""

# ============================================
# SAVE DATASETS
# ============================================

members.to_csv("../data/members.csv", index=False)
checkins.to_csv("../data/checkins.csv", index=False)
payments.to_csv("../data/payments.csv", index=False)

print("Datasets successfully saved to /data directory.")

Datasets successfully saved to /data directory.
